In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

# Carga de los ficheros con los indices de gini y los indicadores P20P80

Se han estructurado los datos en la carpeta de inputs para la dimensión socioeconómica de forma que se dispone de un fichero CSV a nivel
de provincia. Así pues, se han definido una función en las utilidades que se encargan de cargar y dar una limpieza inicial a los datos.

Los datos se obtienen del atlas de distribución de renta de los hogares:
https://www.ine.es/dynt3/inebase/index.htm?padre=12385&capsel=12384

In [4]:
path = os.path.join(DATA_INPUTS_DS, "Indice de gini y distribucion de renta 20-80")

indicadores = carga_datos_ine(path)

# Veo una muestra de su estructura y contenido
print(indicadores.info())
indicadores.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 64278 entries, 36 to 147311
Data columns (total 7 columns):
 #   Column                                             Non-Null Count  Dtype 
---  ------                                             --------------  ----- 
 0   Municipios                                         64278 non-null  object
 1   Distritos                                          64278 non-null  object
 2   Secciones                                          64278 non-null  object
 3   Índice de Gini y Distribución de la renta P80/P20  64278 non-null  object
 4   Periodo                                            64278 non-null  int64 
 5   Total                                              50913 non-null  object
 6   Provincia                                          64278 non-null  object
dtypes: int64(1), object(6)
memory usage: 3.9+ MB
None


,Municipios,Distritos,Secciones,Índice de Gini y Distribución de la renta P80/P20,Periodo,Total,Provincia
44559,24089 León,2408903 León distrito 03,2408903012 León sección 03012,Distribución de la renta P80/P20,2023,"2,4",Leon
126051,47186 Valladolid,4718604 Valladolid distrito 04,4718604030 Valladolid sección 04030,Distribución de la renta P80/P20,2017,"2,4",Valladolid
92892,40053 Cerezo de Abajo,4005301 Cerezo de Abajo distrito 01,4005301001 Cerezo de Abajo sección 01001,Distribución de la renta P80/P20,2020,"2,8",Segovia
42212,24054 Cimanes de la Vega,2405401 Cimanes de la Vega distrito 01,2405401001 Cimanes de la Vega sección 01001,Índice de Gini,2021,"27,0",Leon
52889,24191 Vallecillo,2419101 Vallecillo distrito 01,2419101001 Vallecillo sección 01001,Índice de Gini,2018,"27,7",Leon


# Estandarización del dataframe de datos del INE

Como se puede observar, el fichero csv de datos del INE tiene un formato poco amigable para el tratamiento de los datos. En lugar de tener una fila
por cada par sección-año y varias columnas (una por factor), tiene múltiples filas con distintos indicadores para una misma sección, lo que resulta
complejo de tratar. Además, se observa como se mezcla el código del municipio, distrito y seccion con el texto, y deberían tener una columna con
los códigos.

In [5]:
indicadores_estandarizados = estandarizar_df_ine(indicadores, "Índice de Gini y Distribución de la renta P80/P20")
print(indicadores_estandarizados.info())
indicadores_estandarizados.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 64278 entries, 36 to 147311
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Provincia  64278 non-null  object
 1   CMuni      64278 non-null  object
 2   CUSEC      64278 non-null  object
 3   Indicador  64278 non-null  object
 4   Periodo    64278 non-null  int64 
 5   Total      50913 non-null  object
dtypes: int64(1), object(5)
memory usage: 3.4+ MB
None


,Provincia,CMuni,CUSEC,Indicador,Periodo,Total
82052,Salamanca,37274,3727401002,Índice de Gini,2015,"36,4"
8874,Avila,05160,0516001001,Índice de Gini,2023,"25,4"
100782,Segovia,40194,4019407003,Índice de Gini,2023,"29,6"
117835,Valladolid,47073,4707301001,Índice de Gini,2016,"31,4"
83912,Salamanca,37274,3727406001,Distribución de la renta P80/P20,2018,"2,8"


## Filtrado de años
Revisando la documentación del INE, en el año 2021 se cambió radicalemente la metodología que define las secciones censales,
y en concreto en Castilla y León se aumentó el numero de censos de 2700 a unos 3500 apróximadamente. Es por ello que, si bien
se dispone de datos de años anteriores, sería complejo y peligroso fragmentar y proyectar los censos de años previos en la malla 
censal actual, por lo que se filtraran datos de años previos

In [6]:
# Reviso los indicadores disponibles
revisar_indicadores_disponibles(indicadores_estandarizados)

# Filtro por los años 2021 - 2023.
indicadores_recientes = indicadores_estandarizados[
    indicadores_estandarizados["Periodo"].isin([2021, 2022, 2023])
].copy()

📅 Años disponibles:
[2023 2022 2021 2020 2019 2018 2017 2016 2015]
------------------------------------------------------------
🧩 Indicadores demográficos disponibles:
  - Índice de Gini
  - Distribución de la renta P80/P20
------------------------------------------------------------


In [7]:
# Pivoto los indicadores para tener una columna por indicador y reducir las filas de la tabla
indicadores_por_seccion = pivotar_indicadores(indicadores_recientes)
print(indicadores_por_seccion.info())
indicadores_por_seccion.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8390 entries, 0 to 8389
Data columns (total 6 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   Provincia                         8390 non-null   object
 1   CMuni                             8390 non-null   object
 2   CUSEC                             8390 non-null   object
 3   Periodo                           8390 non-null   int64 
 4   Distribución_de_la_renta_P80_P20  8390 non-null   object
 5   Índice_de_Gini                    8390 non-null   object
dtypes: int64(1), object(5)
memory usage: 393.4+ KB
None


,Provincia,CMuni,CUSEC,Periodo,Distribución_de_la_renta_P80_P20,Índice_de_Gini
36,Avila,05014,0501402002,2022,"2,7","30,2"
7267,Valladolid,47186,4718610043,2023,"2,2","24,9"
6203,Valladolid,47058,4705801001,2021,"2,9","31,5"
5297,Segovia,40078,4007801001,2022,"2,5","30,9"
5089,Salamanca,37369,3736901001,2021,"3,5","39,3"


# Export de los resultados

In [8]:
# Creamos la carpeta si no existe
os.makedirs(DATA_OUTPUTS_DS, exist_ok=True)

# Rutas de salida
ruta_seccion = os.path.join(DATA_OUTPUTS_DS, "Gini_P20P80_por_seccion.csv")

# Guardar DataFrames
indicadores_por_seccion.to_csv(
    ruta_seccion,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)

print(f"✅ Archivos guardados correctamente en: {DATA_OUTPUTS_DS}")

✅ Archivos guardados correctamente en: D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\DS_Dim_socioeconomica
